In [212]:
import numpy as np

### Speicfy setup of qubits

In [213]:
rng = np.random.default_rng(1234)

In [214]:
# Basic utilities
def normalise(psi):
    norm = np.linalg.norm(psi)
    if norm == 0:
        return psi
    return psi / norm

def dagger(psi):
    return np.conjugate(psi.T)

def density_matrix(psi):
    psi = normalise(psi)
    return np.outer(psi, dagger(psi))

def tensor_product(A,B):
    return np.kron(A,B)

def depolarise(rho, p): # alters the density matrix rho with depolarising noise p
    d = rho.shape[0]
    I = np.eye(d)
    return (1 - p) * rho + p * I / d

state_0 = np.array([1,0])
state_1 = np.array([0,1])


#### Trial 2-qubit state

In [215]:
# Use bell state as a trial

state_bell = normalise(tensor_product(state_0, state_0) + tensor_product(state_1, state_1))
rho_bell = density_matrix(state_bell)

In [216]:
print(rho_bell)
print(np.trace(rho_bell))  # Should be 1

[[0.5 0.  0.  0.5]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.5 0.  0.  0.5]]
1.0000000000000002


Define POVM elements to measure in Z basis

In [217]:
Z = [[1,0],[0,-1]]
ZZ = tensor_product(Z,Z)
print(ZZ)

[[ 1  0  0  0]
 [ 0 -1  0  0]
 [ 0  0 -1  0]
 [ 0  0  0  1]]


Can see from the ZZ matrix that the eigenvalues are +1 (for |00> and |11>) and -1 (for |10> and |01>)

In tomography, you aim to reconstruct the density matrix. Using ZZ gives an eigenvalue [-1,1] simply saying how likely the qubits are to be aligned or misaligned. Projectors (such as |00><00|) give individual outcomes of the ZZ measurement (the entire joint probability distribution).

Projectors and density matrices are visually and mathematically identical, but projectors are used to measure states and density matrices represent prepared states

The POVM is a list of the projectors defined by the eigenstates of the operator, ZZ in this case. This POVM is related to ZZ as we are using its eigenstates as the projectors.

In [218]:
E_00 = density_matrix(tensor_product(state_0, state_0))  # |00><00|
E_01 = density_matrix(tensor_product(state_0, state_1))  # |01><01|
E_10 = density_matrix(tensor_product(state_1, state_0))  # |10><10|
E_11 = density_matrix(tensor_product(state_1, state_1))  # |11><11|

POVM_ZZ = [E_00, E_01, E_10, E_11]

In [219]:
probs = [float(np.trace(rho_bell @ E)) for E in POVM_ZZ]
# Finds probabilities of measuring each outcome in the POVM given the state rho_bell

In [220]:
print(probs)

[0.5000000000000001, 0.0, 0.0, 0.5000000000000001]


Probabilities have come out as expected for Bell State

### Constructing POVMs using X, Y, and Z basis

In [221]:
# The three Pauli matrices

X = np.array([[0, 1],
              [1, 0]], dtype=complex)

Y = np.array([[0, -1j],
              [1j, 0]], dtype=complex)

Z = np.array([[1, 0],
              [0, -1]], dtype=complex)

In [222]:
# States for X basis
states_X = np.linalg.eig(X)[1]
states_Y = np.linalg.eig(Y)[1]
states_Z = np.linalg.eig(Z)[1]

In [223]:
# Define a function to get POVM from two sets of basis states
def get_POVM(A,B):
    POVM = []
    E_0 = density_matrix(tensor_product(A[0], B[0]))
    E_1 = density_matrix(tensor_product(A[0], B[1]))
    E_2 = density_matrix(tensor_product(A[1], B[0]))
    E_3 = density_matrix(tensor_product(A[1], B[1]))
    POVM = [E_0, E_1, E_2, E_3]
    return POVM
# Note for future that density_matrix automatically normalises the states

In [224]:
POVM_XX = get_POVM(states_X, states_X)
POVM_XY = get_POVM(states_X, states_Y)
POVM_XZ = get_POVM(states_X, states_Z)
POVM_YX = get_POVM(states_Y, states_X)
POVM_YY = get_POVM(states_Y, states_Y)
POVM_YZ = get_POVM(states_Y, states_Z)
POVM_ZX = get_POVM(states_Z, states_X)
POVM_ZY = get_POVM(states_Z, states_Y)
POVM_ZZ = get_POVM(states_Z, states_Z)

In [225]:
# Check that POVMs sum to identity
print(sum(POVM_XX).round(6))

[[ 1.+0.j  0.+0.j -0.+0.j  0.+0.j]
 [ 0.+0.j  1.+0.j  0.+0.j  0.+0.j]
 [-0.+0.j  0.+0.j  1.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  1.+0.j]]


In [226]:
def get_probs(rho, POVM):
    ps = np.array([float(np.real(np.trace(rho @ E))) for E in POVM])
    ps = np.clip(ps, 1e-15, 1.0)
    ps /= np.sum(ps)  # Normalise
    return ps

In [227]:
# introduce noise to rho_bell
rho_bell_noisy = depolarise(rho_bell, 0.1)

In [228]:
probs_XX = get_probs(rho_bell_noisy, POVM_XX)
probs_XY = get_probs(rho_bell_noisy, POVM_XY)
probs_XZ = get_probs(rho_bell_noisy, POVM_XZ)
probs_YX = get_probs(rho_bell_noisy, POVM_YX)
probs_YY = get_probs(rho_bell_noisy, POVM_YY)
probs_YZ = get_probs(rho_bell_noisy, POVM_YZ)
probs_ZX = get_probs(rho_bell_noisy, POVM_ZX)
probs_ZY = get_probs(rho_bell_noisy, POVM_ZY)
probs_ZZ = get_probs(rho_bell_noisy, POVM_ZZ)

**We constructed all nine two-qubit product-Pauli POVMs (XX, XY, …, ZZ) and used them to compute, via Born’s rule, the exact theoretical measurement probabilities for each outcome of the Bell state.**

### Generating noisy bell state shots - aim to reconstruct Bell State density matrix from these

In [229]:
# Generate with gaussian noise to start with

def generate_noisy_shot(probs, noise_level):
    noisy_probs = probs + rng.normal(0, noise_level, size=len(probs))
    noisy_probs = np.clip(noisy_probs, 0, None)  # Ensure no negative probabilities
    noisy_probs /= np.sum(noisy_probs)  # Renormalise
    shot = rng.multinomial(1, noisy_probs) # Draw probabilistic shots
    return shot
# Noise is remade for each shot to simulate noise in a real experiment

In [230]:
# trial for ZZ POVM
N_shots = 100
noise_level = 0.2  # Standard deviation of Gaussian noise
shots_ZZ = [generate_noisy_shot(probs_ZZ, noise_level) for _ in range(N_shots)]

In [231]:
measured_probs_ZZ = np.mean(shots_ZZ, axis=0)
print("Theoretical probs ZZ:", probs_ZZ)
print("Measured probs ZZ:", measured_probs_ZZ)

Theoretical probs ZZ: [0.475 0.025 0.025 0.475]
Measured probs ZZ: [0.37 0.1  0.07 0.46]


In [232]:
shots_XX = [generate_noisy_shot(probs_XX, noise_level) for _ in range(N_shots)]
shots_XY = [generate_noisy_shot(probs_XY, noise_level) for _ in range(N_shots)]
shots_XZ = [generate_noisy_shot(probs_XZ, noise_level) for _ in range(N_shots)]
shots_YX = [generate_noisy_shot(probs_YX, noise_level) for _ in range(N_shots)]
shots_YY = [generate_noisy_shot(probs_YY, noise_level) for _ in range(N_shots)]
shots_YZ = [generate_noisy_shot(probs_YZ, noise_level) for _ in range(N_shots)]
shots_ZX = [generate_noisy_shot(probs_ZX, noise_level) for _ in range(N_shots)]
shots_ZY = [generate_noisy_shot(probs_ZY, noise_level) for _ in range(N_shots)]

### Putting things in place to reconstruct density matrix

Use Bayes theroem to reconstruct the state

In [233]:
POVMs = [POVM_XX, POVM_XY, POVM_XZ, POVM_YX, POVM_YY, POVM_YZ, POVM_ZX, POVM_ZY, POVM_ZZ]

In [234]:
counts_XX = np.sum(np.asarray(shots_XX), axis=0)
counts_XY = np.sum(np.asarray(shots_XY), axis=0)
counts_XZ = np.sum(np.asarray(shots_XZ), axis=0)
counts_YX = np.sum(np.asarray(shots_YX), axis=0)
counts_YY = np.sum(np.asarray(shots_YY), axis=0)
counts_YZ = np.sum(np.asarray(shots_YZ), axis=0)
counts_ZX = np.sum(np.asarray(shots_ZX), axis=0)
counts_ZY = np.sum(np.asarray(shots_ZY), axis=0)
counts_ZZ = np.sum(np.asarray(shots_ZZ), axis=0)

In [235]:
counts = [counts_XX, counts_XY, counts_XZ, counts_YX, counts_YY, counts_YZ, counts_ZX, counts_ZY, counts_ZZ]
# Corresponds to the POVMs list order

In [236]:
# Use log likelihood to remove factorials, constant in rho
def log_likelihood(rho, POVMs, counts):
    ll = 0
    for POVM, count in zip(POVMs, counts):
        probs = get_probs(rho, POVM)
        ll += np.sum(count * np.log(probs))
    return ll

In [237]:
# We need a prior, draw random matrix using Ginibre
# Guess lots of priors and find one with the highest likelihood
def random_density_hs():
    G = (np.random.randn(4,4) + 1j*np.random.randn(4,4))/np.sqrt(2.0)
    X = G @ G.conj().T
    return X / X.trace().real

In [238]:
# need to define fidelity between reconstructed solution (rho) and true bell matrix (sigma)
# F = (Trace sqrt( sqrt(sigma) * rho * sqrt(sigma) ) )^2
def fidelity(rho, sigma):
    vals, vecs = np.linalg.eigh(sigma)
    sqrt_sigma = (vecs * np.sqrt(np.clip(vals,0,1))) @ vecs.conj().T # finding root sigma from eigendecomposition
    M = sqrt_sigma @ rho @ sqrt_sigma
    vals_M, _ = np.linalg.eigh((M + M.conj().T) / 2)  # Ensure Hermitian
    return (np.sum(np.sqrt(vals_M)))**2 # Gives final fidelity

In [239]:
# Define weighted percentiles given how likely each solution is
# Some represent a larger fraction of the posterior probability 
def weighted_percentile(values, weights, quantiles):
    order = np.argsort(values)
    v = np.asarray(values)[order]
    w = np.asarray(weights)[order]
    cdf = np.cumsum(w)
    return np.interp(np.array(quantiles), cdf, v)

In [240]:
rhos = []
log_likes = np.empty(20000, float)

for i in range(20000):
    rho = random_density_hs()
    rhos.append(rho)
    log_likes[i] = log_likelihood(rho, POVMs, counts)
    # Finding log likelihood for the randomly selected priors

max_ll = np.max(log_likes)
weights = np.exp(log_likes - max_ll)
weights /= weights.sum()

Posterior distribution p(rho|D) proportional to p(D|rho)p(rho)
p(rho|D) is probability of rho given data
p(D|rho) is probability of data given rho (likelihood)
p(rho) is the prior

In [241]:
# Get posterior mean density matrix - average rho weighted by likelihood
rho_mean = np.zeros((4,4), complex)
for weight, rho in zip(weights, rhos):
    rho_mean += weight * rho

In [242]:
rho_best = rhos[int(np.argmax(log_likes))]

rho_phi = rho_bell
fidelities = [fidelity(rho, rho_phi) for rho in rhos]
mean_fidelity = np.sum(np.array(fidelities) * weights)
fidelity_credible_interval = weighted_percentile(fidelities, weights, [0.05, 0.95])

/var/folders/fx/lnm0r8px7dl3hm8pdd5ryl4w0000gn/T/ipykernel_26204/679029461.py:8: RuntimeWarning: invalid value encountered in sqrt
  return (np.sum(np.sqrt(vals_M)))**2 # Gives final fidelity


In [243]:
np.set_printoptions(precision=6, suppress=True)
print("Posterior mean ρ:\n", rho_mean)
print("\nMAP-ish ρ (best sample):\n", rho_best)
print("True Bell state ρ:\n", rho_bell)
print("Fidelity to |Φ+> (mean, 95% CI):", mean_fidelity, fidelity_credible_interval)

Posterior mean ρ:
 [[ 0.355168-0.j       -0.058246+0.026415j  0.010533+0.029722j
   0.230264-0.059418j]
 [-0.058246-0.026415j  0.138943+0.j        0.003251-0.017717j
  -0.026209-0.054555j]
 [ 0.010533-0.029722j  0.003251+0.017717j  0.088354+0.j
   0.059923+0.03704j ]
 [ 0.230264+0.059418j -0.026209+0.054555j  0.059923-0.03704j
   0.417535-0.j      ]]

MAP-ish ρ (best sample):
 [[ 0.328664+0.j       -0.090224+0.031997j  0.043972+0.05393j
   0.238748-0.017809j]
 [-0.090224-0.031997j  0.153147+0.j       -0.036771-0.002288j
  -0.064045+0.00084j ]
 [ 0.043972-0.05393j  -0.036771+0.002288j  0.087867+0.j
   0.061549+0.055496j]
 [ 0.238748+0.017809j -0.064045-0.00084j   0.061549-0.055496j
   0.430323-0.j      ]]
True Bell state ρ:
 [[0.5 0.  0.  0.5]
 [0.  0.  0.  0. ]
 [0.  0.  0.  0. ]
 [0.5 0.  0.  0.5]]
Fidelity to |Φ+> (mean, 95% CI): nan [0.59626  0.621009]


Posterior mean gives closest average to true bell state

In [244]:
print(fidelity(rho_best, rho_bell))
print(fidelity(rho_mean, rho_bell))
fidelity(rho_bell, rho_bell)

0.6182413540193658
0.6166153943602858


np.float64(0.9999999999999998)

### Using a NN to find the final state

In [247]:
import torch
import torch.nn as nn
import torch.optim as optim

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [250]:
POVMs = POVMs
counts = counts
# use the same counts and POVMs as before

counts has 9 lists (each povm setting) with 4 outcomes each. This corresponds to 36 starting nodes

In [254]:
counts_lst = []
for count in counts:
    count = np.array(count)
    counts_lst.append(count)
counts = np.concatenate(counts_lst, axis=0)

Aiming for 4x4 density matrix, so input is 36, output is 16

In [258]:
model = nn.Sequential(
    nn.Linear(36, 128), # 36 input
    nn.ReLU(),
    nn.Linear(128, 64), # 128 hidden, then 64 hidden
    nn.ReLU(),
    nn.Linear(64, 16), # 16 output
).to(DEVICE)

In [259]:
# Density matrix must be hermitian, positive semidefinite, trace 1
# Must enforce these to construct rho from NN output
def rho_from_output(theta):
    # Reshape output to 4x4 matrix
    T = theta.view(4, 4)
    # Construct rho = T†T / Tr(T†T)
    rho = T.conj().T @ T
    rho /= torch.trace(rho)
    return rho

#### Need to simulate examples to train the NN

In [260]:
def simulate_example(N_shots=200, noise_level=0.01):
    # random true state: 50% depolarised Bell, 50% random (variety helps)
    if np.random.rand() < 0.5:
        # assumes you already defined rho_bell above
        p = np.random.uniform(0.0, 0.2)
        rho_true = rho_bell_noisy

    counts_per_setting = []

    for POVM in POVMs:
        ps = get_probs(rho_true, POVM)
        # add small Gaussian noise to probs
        ps = ps + np.random.normal(0, noise_level, size=ps.shape)
        ps = np.clip(ps, 1e-12, None); ps /= ps.sum()
        counts = np.random.multinomial(N_shots, ps).astype(np.float32)
        counts_per_setting.append(counts)

    c = np.concatenate(counts_per_setting, axis=0).astype(np.float32)
    return c, rho_true

In [261]:
def build_dataset(N):
    C = np.zeros((N, 36), dtype=np.float32)
    for i in range(N):
        c, _ = simulate_example(N_shots=200, noise_level=0.01)
        C[i] = c
    return C

In [264]:
povm_tensors = [torch.tensor(np.stack(POVM), device=DEVICE) for POVM in POVMs]

## NEED TO TRAIN ON THE SYNTHETIC DATA THEN VALIDATE FOR BELL STATE